# 11 · Optional public-data plotting examples

**ScGeo 0.2.0 learning track.** Use existing checksum-pinned public PBMC3k and/or preprocessed pancreas AnnData. No download occurs automatically. PBMC annotation mapping is internal to the same dataset, not independent validation; pancreas vectors are supplied upstream, not refitted. No treatment or sample IDs are invented.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "11_public_data_optional")

## Configure explicitly supplied public data

Set either environment variable before starting the kernel, or edit the path dictionary in the next cell. If neither is set, the notebook records that the public-data exercise was not run. File checks are inherited from the release's [public-data gallery](https://github.com/liuifrec/scgeo/blob/v0.2.0/docs/REAL_DATA_PLOTTING.md).

In [ ]:
import os
from real_data_plotting_gallery import SOURCES, prepare_dataset, render_pbmc, render_pancreas, validation_summary
from embedding_facets_gallery import run as facet_gallery
from geometry_dynamics_atlas_gallery import run_gallery as atlas_gallery
paths = {"pbmc": os.environ.get("SCGEO_TUTORIAL_PBMC_H5AD"),
         "pancreas": os.environ.get("SCGEO_TUTORIAL_PANCREAS_H5AD")}
paths = {name: Path(path).expanduser() for name, path in paths.items() if path}
display(pd.DataFrame([{"dataset":name,"sha256":spec["sha256"],"size_bytes":spec["size_bytes"],
                       "url":spec["url"],"provided":name in paths} for name,spec in SOURCES.items()]))

## Prepare, render and audit only the provided files

This exercise uses the release's exact checksum verification and public-data renderers. It does not access manuscript data, fit new velocities, or perform an irradiation contrast.

In [ ]:
exports = {}
if not paths:
    print("NOT RUN: no checksum-pinned public files supplied. Synthetic notebooks remain runnable.")
    status = {"status":"not_run","reason":"No public file paths provided","datasets":[]}
else:
    prepared = out / "prepared"
    figures = out / "real_gallery"
    figures.mkdir()
    for dataset, path in paths.items():
        prepare_dataset(dataset, path, prepared, download=False)
        if dataset == "pbmc":
            exports[dataset] = render_pbmc(prepared, figures, FORMATS)
        else:
            exports[dataset] = {"velocity":render_pancreas(prepared, figures, FORMATS)}
        facet_gallery(out / f"{dataset}_facets", dataset=dataset,
                      input_h5ad=path, download_public=False, formats=FORMATS)
    audit = validation_summary(exports, prepared, figures)
    atlas_gallery(out / "atlas", formats=FORMATS, **paths)
    show_gallery(figures, limit=3)
    status = {"status":"executed","datasets":list(paths),"audit":str(figures)}
(out / "public_data_status.json").write_text(json.dumps(status, indent=2) + "\n")
display(status)

**Interpretation:** these are descriptive rendering and software checks. Neither supplied dataset alone establishes a treatment effect, temporal transition, or independent annotation accuracy.